In [11]:
import sys
sys.path.insert(0, '.')

# Add Supplier Homepages

Reads the supplier overview URLs from `data/supplier_products_overview_pages.txt`, extracts the root domain as the homepage, and writes it to the `Homepage` column in the `Supplier` table via `db.set_supplier_homepage()`.

Matching is done by normalising the domain/path segments and using `db.search_supplier()` (case-insensitive, punctuation-insensitive). Logs a warning if no supplier is found or if the match is ambiguous.

In [13]:
from urllib.parse import urlparse
from db import search_supplier, set_supplier_homepage

LIST_PATH = "../data/supplier_products_overview_pages.txt"

with open(LIST_PATH) as f:
    urls = [line.strip() for line in f if line.strip()]

for url in urls:
    parsed = urlparse(url)
    homepage = f"{parsed.scheme}://{parsed.hostname}"

    # build search terms: domain parts then path segments
    parts = parsed.hostname.split(".")
    if parts[0] in ("www", "eu", "us", "uk"):
        parts = parts[1:]
    if parts[-1] in ("com", "org", "net", "io"):
        parts = parts[:-1]
    domain_term = "".join(parts)

    path_terms = [p for p in parsed.path.split("/") if p]
    search_terms = [domain_term] + path_terms

    matches = []
    for term in search_terms:
        matches = search_supplier(term)
        if len(matches) == 1:
            break

    if len(matches) == 0:
        print(f"WARNING: no supplier found for {url}")
    elif len(matches) > 1:
        ids = [(m["Id"], m["Name"]) for m in matches]
        print(f"WARNING: ambiguous match for {url} → {ids}")
    else:
        supplier = matches[0]
        set_supplier_homepage(supplier["Id"], homepage)
        print(f"Set {supplier['Name']} (id={supplier['Id']}) homepage → {homepage}")

Set BulkSupplements (id=7) homepage → https://www.bulksupplements.com
Set Capsuline (id=9) homepage → https://eu.capsuline.com
Set Custom Probiotics (id=12) homepage → https://www.customprobiotics.com
Set Nutra Blend (id=24) homepage → https://feedsforless.com
Set PureBulk (id=28) homepage → https://purebulk.com
Set Source-Omega LLC (id=31) homepage → https://www.source-omega.com
Set Spectrum Chemical (id=33) homepage → https://www.spectrumchemical.com
Set Trace Minerals (id=37) homepage → https://www.traceminerals.com


In [14]:
# verify results
from db import get_suppliers
[(s["Name"], s.get("Homepage")) for s in get_suppliers() if s.get("Homepage")]

[('BulkSupplements', 'https://www.bulksupplements.com'),
 ('Capsuline', 'https://eu.capsuline.com'),
 ('Custom Probiotics', 'https://www.customprobiotics.com'),
 ('Nutra Blend', 'https://feedsforless.com'),
 ('PureBulk', 'https://purebulk.com'),
 ('Source-Omega LLC', 'https://www.source-omega.com'),
 ('Spectrum Chemical', 'https://www.spectrumchemical.com'),
 ('Trace Minerals', 'https://www.traceminerals.com')]